# Edge-Enhanced GraphSAGE — Live Demo (Colab edition)

**Component:** Member 1 — Relational Fraud Detector
**Stage 3a model**: F1=0.5387, AUROC=0.9497, Recall=0.5663

This notebook runs the live fraud-detection demo. It auto-detects whether you're
running on Google Colab or locally and sets up paths accordingly.

**Before running on Colab, upload these files to your Drive:**
- `/My Drive/R26-IT-121-data/paysim_graph.pt`  (195 MB)
- `/My Drive/R26-IT-121-data/stage3a_focal.pt`  (45 KB)


## 1. Environment setup (auto-detects Colab vs local)


In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print('Environment:', 'Google Colab' if IS_COLAB else 'Local')


### 1a. Colab-only: clone repo, install dependencies, mount Drive

If running locally, this whole cell is skipped.


In [ ]:
if IS_COLAB:
    print('Cloning repo...')
    if not Path('/content/R26-IT-121').exists():
        !git clone -q https://github.com/LEXES7/R26-IT-121.git /content/R26-IT-121
    %cd /content/R26-IT-121/GraphSage

    print('Installing dependencies (~2 min)...')
    !pip install -q torch-geometric pyarrow
    !pip install -q -e .

    # Add the package to sys.path so imports work
    sys.path.insert(0, '/content/R26-IT-121/GraphSage/src')

    # Mount Drive for the data files
    print('Mounting Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')

    # Symlink the data files into the expected paths
    GRAPH_SRC = Path('/content/drive/MyDrive/R26-IT-121-data/paysim_graph.pt')
    CKPT_SRC  = Path('/content/drive/MyDrive/R26-IT-121-data/stage3a_focal.pt')
    GRAPH_DST = Path('/content/R26-IT-121/GraphSage/data/graph/paysim_graph.pt')
    CKPT_DST  = Path('/content/R26-IT-121/GraphSage/checkpoints/stage3a_focal.pt')
    GRAPH_DST.parent.mkdir(parents=True, exist_ok=True)
    CKPT_DST.parent.mkdir(parents=True, exist_ok=True)
    if not GRAPH_DST.exists() and GRAPH_SRC.exists():
        os.symlink(GRAPH_SRC, GRAPH_DST)
    if not CKPT_DST.exists() and CKPT_SRC.exists():
        os.symlink(CKPT_SRC, CKPT_DST)
    print('Setup complete.')
    REPO = Path('/content/R26-IT-121/GraphSage')
else:
    REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

GRAPH_PATH = REPO / 'data' / 'graph' / 'paysim_graph.pt'
CKPT_PATH = REPO / 'checkpoints' / 'stage3a_focal.pt'
assert GRAPH_PATH.exists(), f'Graph file not found at {GRAPH_PATH}'
assert CKPT_PATH.exists(), f'Checkpoint not found at {CKPT_PATH}'
print(f'Graph: {GRAPH_PATH.stat().st_size // 1024**2} MB at {GRAPH_PATH}')
print(f'Ckpt:  {CKPT_PATH.stat().st_size // 1024} KB at {CKPT_PATH}')


## 2. Load graph + Stage 3a model


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import k_hop_subgraph

from graphsage.models.edge_sage import EdgeEnhancedGraphSAGE
from graphsage.training.threshold_tuning import metrics_at_threshold, find_best_threshold_for_f1

data = torch.load(GRAPH_PATH, weights_only=False)
print(data)
print(f'Total mules: {int(data.y.sum().item()):,}  |  Test mules: {int((data.test_mask & data.y.bool()).sum().item()):,}')


In [ ]:
ckpt = torch.load(CKPT_PATH, weights_only=False, map_location='cpu')
hp = ckpt['hyperparameters']
model = EdgeEnhancedGraphSAGE(
    in_dim=int(data.x.shape[1]),
    edge_dim=int(data.edge_attr.shape[1]),
    hidden_dim=hp.get('hidden_dim', 64),
    edge_mlp_hidden=hp.get('edge_mlp_hidden', 32),
    dropout=hp.get('dropout', 0.3),
)
model.load_state_dict(ckpt['state_dict'])
model.eval()
print(f'Model loaded - {sum(p.numel() for p in model.parameters()):,} parameters')


## 3. Run inference on full graph + threshold tuning


In [ ]:
with torch.no_grad():
    logits, attentions = model.forward_with_attention(
        data.x, data.edge_index, data.edge_attr
    )
    probs = torch.sigmoid(logits)

best_thresh, val_f1 = find_best_threshold_for_f1(
    logits[data.val_mask], data.y[data.val_mask]
)
print(f'Best threshold (from val): {best_thresh:.4f}  (val F1 = {val_f1:.4f})')

test_m = metrics_at_threshold(logits[data.test_mask], data.y[data.test_mask], best_thresh)
print(f'\nTest metrics @ tuned threshold:')
for k, v in test_m.items():
    print(f'  {k:12s}: {v:.4f}')


## 4. Pick a real fraud node + extract suspicious subgraph


In [ ]:
test_mules = (data.test_mask & data.y.bool()).nonzero(as_tuple=True)[0]
top_idx = test_mules[probs[test_mules].argmax()].item()
top_prob = float(probs[top_idx].item())
print(f'Selected node {top_idx}: predicted P(fraud)={top_prob:.4f} (ground truth: MULE)')

subset, sub_ei, mapping, edge_mask = k_hop_subgraph(
    node_idx=top_idx, num_hops=2, edge_index=data.edge_index,
    relabel_nodes=True, num_nodes=data.num_nodes,
)
sub_attentions = attentions[1][edge_mask]
sub_y = data.y[subset]
sub_probs = probs[subset]
flagged_local = (subset == top_idx).nonzero(as_tuple=True)[0].item()

print(f'\nSuspicious subgraph (k=2): {len(subset):,} nodes, {sub_ei.shape[1]:,} edges')
print(f'  Mules in subgraph: {int(sub_y.sum().item())}')


## 5. Visualize the mule ring (Edge-MLP attention drives edge thickness)


In [ ]:
MAX_NODES = 30
if len(subset) > MAX_NODES:
    top_edges_idx = sub_attentions.argsort(descending=True)[:MAX_NODES]
    keep_edges = sub_ei[:, top_edges_idx]
    keep_nodes = set(keep_edges[0].tolist()) | set(keep_edges[1].tolist())
    keep_nodes.add(flagged_local)
    kept_mask = torch.tensor([(int(s) in keep_nodes) and (int(d) in keep_nodes)
                              for s, d in sub_ei.t()])
    plot_ei = sub_ei[:, kept_mask]
    plot_attn = sub_attentions[kept_mask]
else:
    plot_ei = sub_ei
    plot_attn = sub_attentions
    keep_nodes = set(range(len(subset)))

G = nx.DiGraph()
for i in keep_nodes:
    G.add_node(i, is_mule=int(sub_y[i].item()), prob=float(sub_probs[i].item()))
for col, (src, dst) in enumerate(plot_ei.t().tolist()):
    G.add_edge(src, dst, weight=float(plot_attn[col].item()))

node_colors, node_sizes = [], []
for n in G.nodes():
    if n == flagged_local:
        node_colors.append('crimson'); node_sizes.append(900)
    elif G.nodes[n]['is_mule']:
        node_colors.append('orangered'); node_sizes.append(500)
    elif G.nodes[n]['prob'] > 0.5:
        node_colors.append('orange'); node_sizes.append(350)
    else:
        node_colors.append('lightsteelblue'); node_sizes.append(250)
edge_widths = [2 + 6 * G.edges[e]['weight'] for e in G.edges()]
edge_colors = ['crimson' if G.edges[e]['weight'] > 0.7 else 'gray' for e in G.edges()]

fig, ax = plt.subplots(figsize=(13, 9))
pos = nx.spring_layout(G, seed=42, k=1.2)
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, ax=ax)
nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=edge_widths,
                       arrows=True, arrowsize=14, ax=ax, alpha=0.7)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title(f'Suspicious Subgraph - flagged node {top_idx}\n'
             f'Crimson center = mule. Edge thickness = Edge-MLP attention.',
             fontsize=12)
ax.axis('off')
plt.tight_layout(); plt.show()


## 6. Build the JSON payload sent to Member 4's Fusion Engine


In [ ]:
import json
in_deg = int((sub_ei[1] == flagged_local).sum().item())
out_deg = int((sub_ei[0] == flagged_local).sum().item())
fresh_count = sum(1 for n in range(len(subset)) if int(data.x[subset[n], 1].item()) <= 1)
fresh_ratio = fresh_count / max(len(subset), 1)
mean_drain = float(data.edge_attr[edge_mask][:, 1].mean().item())
mules_count = int(sub_y.sum().item())

response = {
    'transaction_id': f'TX_DEMO_{top_idx}',
    'model_version': 'graphsage-edge-mlp-focal-v0.3.0',
    'stage': 'stage_3a_focal',
    'relational_risk_score': round(top_prob, 4),
    'risk_level': 'CRITICAL' if top_prob > 0.9 else 'HIGH' if top_prob > 0.5 else 'MEDIUM',
    'confidence': round(max(top_prob, 1 - top_prob), 4),
    'suspicious_subgraph': {
        'k_hop': 2,
        'node_count': int(len(subset)),
        'edge_count': int(sub_ei.shape[1]),
        'sink_account': f'NODE_{top_idx}',
        'pattern': 'HUB_AND_SPOKE',
        'pattern_confidence': round(min(in_deg / 5.0, 1.0), 3),
        'structural_evidence': {
            'flagged_in_degree': in_deg,
            'flagged_out_degree': out_deg,
            'fresh_sender_ratio': round(fresh_ratio, 3),
            'mean_drain_ratio': round(mean_drain, 3),
            'mules_in_subgraph': mules_count,
        },
    },
    'metadata': {
        'tuned_threshold': round(best_thresh, 4),
        'extraction_method': 'k_hop_subgraph_pyg',
    },
}
print(json.dumps(response, indent=2))


## 7. Summary

✅ Live inference running on **Google Colab**, no local install needed
✅ Trained Stage 3a model produces real fraud predictions
✅ Edge-MLP attention (Novelty 1) drives edge thickness in the visualization
✅ k=2 hop subgraph extraction (Novelty 3) for the forensic view
✅ JSON payload generated in the locked schema for Member 4

**This Colab notebook is the demo backup for May 11.**
Share this link → anyone can click and run.
